In [0]:
import pyspark.sql.functions as F
import random

In [0]:
%sql
SELECT *
FROM workspace.notebook_breweries.bronze_breweries

In [0]:
%sql
SELECT * 
FROM workspace.notebook_breweries.silver_breweries

In [0]:
%sql
SELECT *
FROM workspace.notebook_breweries.gold_breweries

In [0]:
%sql
SELECT id, phone, ingestion_ts 
FROM notebook_breweries.silver_breweries
WHERE id = '5128df48-79fc-4f0f-8b52-d06be54d0cec'
ORDER BY ingestion_ts


In [0]:
%sql
SELECT id, name, valid_to, COUNT(*) as versioni
FROM workspace.notebook_breweries.gold_breweries
GROUP BY id, name, valid_to
HAVING COUNT(*) >= 1 AND valid_to IS NOT NULL
ORDER BY versioni DESC

In [0]:
%sql
SELECT id, name, COUNT(*) as versioni
FROM workspace.notebook_breweries.silver_breweries
GROUP BY id, name
HAVING COUNT(*) > 1
ORDER BY versioni DESC

In [0]:
%sql
SELECT COUNT(*), MIN(ingestion_ts), MAX(ingestion_ts)
FROM workspace.notebook_breweries.silver_breweries


In [0]:
silver = spark.read.table("workspace.pipeline_breweries.silver_breweries")
print(silver.count())
print(silver.columns)


In [0]:
silver_check = spark.read.table("notebook_breweries.silver_breweries")
print(silver_check.count())

silver_check.groupBy("id").count().filter(F.col("count") > 1).show()


###TEST Notebook

In [0]:
silver = spark.read.table("workspace.notebook_breweries.silver_breweries")
print(silver.count())
print(silver.columns)


In [0]:
#Quante birrerie modificare ad ogni run
NUM_BREWERIES_UPDATE = random.randint(3, 10)

silver = spark.read.table("notebook_breweries.silver_breweries")

#Prendere N record casuali distinti

sample_ids = (silver
                .select("id")
                .distinct()
                .orderBy(F.rand())
                .limit(NUM_BREWERIES_UPDATE)
                .collect()    
            )

print(f"Sample ids raccolti: {len(sample_ids)}")

In [0]:
# Dopo il saveAsTable
result = spark.sql("""
    SELECT COUNT(*), MIN(ingestion_ts), MAX(ingestion_ts)
    FROM workspace.notebook_breweries.silver_breweries
""")
display(result)

###TEST Pipeline

In [0]:
%sql
SELECT *
FROM workspace.notebook_breweries.bronze_breweries

In [0]:
silver = spark.read.table("notebook_breweries.silver_breweries")
print(f"Tabella letta: notebook_breweries.silver_breweries")
print(f"Record: {silver.count()}")

# Simula un inserimento di test
from datetime import datetime
test_row = silver.limit(1).collect()[0].asDict()
test_row["ingestion_ts"] = datetime.now()
test_row["phone"] = "TEST-123"

print(f"Row da inserire: {test_row['id']} - {test_row['phone']} - {test_row['ingestion_ts']}")

test_df = spark.createDataFrame([test_row], schema=silver.schema)
test_df.write.format("delta").mode("append").saveAsTable("notebook_breweries.silver_breweries")

print(f"Record dopo append: {spark.read.table('notebook_breweries.silver_breweries').count()}")
